# LCEL(LangChain Expression Language)
https://reference.langchain.com/python/langchain_core/runnables/

https://reference.langchain.com/python/langchain_core/runnables/?h=runnablelambd#langchain_core.runnables.base.RunnableLambda
  
- LCEL(LangChain Expression Language)은 LangChain에서 체인을 선언적으로 구성할 수 있게 해주는 도메인 특화 언어다.  
- `|` 연산자를 사용해 프롬프트, 모델, 파서 등을 파이프라인처럼 연결한다.

**주요 특징**

- **선언적 문법**: Unix 파이프처럼 `chain = prompt | model | parser` 형태로 직관적이다.  
- **모듈성·유연성**: 프롬프트, LLM, 파서, 검색기, 메모리 등 컴포넌트를 자유롭게 조합할 수 있다.  
- **동기/비동기 지원**: 단일 코드로 동기식·비동기식 실행을 모두 처리할 수 있다.  
- **병렬 처리 최적화**: 병렬 실행 가능한 단계는 자동으로 병렬화해 지연 시간을 줄인다.  
- **고급 기능 기본 제공**:  
  - 스트리밍 출력으로 응답 속도를 향상시킨다.  
  - 실패 시 재시도와 폴백 경로를 설정할 수 있다.  
  - 중간 결과에 접근해 디버깅이나 진행 상황 표시가 가능하다.

**LCEL의 주요 기능**

1. **스트리밍 지원**: 첫 토큰 도달 시간을 단축해 실시간성을 높인다.  
2. **비동기 지원**: asyncio 환경 등 다양한 실행 환경을 동일 코드로 지원한다.  
3. **병렬 실행 최적화**: 병렬화 가능한 단계는 자동으로 분리해 동시에 실행한다.  
4. **재시도·폴백 구성**: 오류 발생 시 지정 횟수만큼 재시도하거나 대체 경로를 실행한다.  
5. **중간 결과 접근**: 최종 출력 이전에 각 단계의 출력을 확인할 수 있다.

**기본 구성 요소**

- **Runnable**: LCEL의 모든 컴포넌트가 상속하는 기본 클래스다.  
- **Chain**: 여러 Runnable을 순차적으로 실행한다.  
- **RunnableMap**: 여러 Runnable을 병렬로 실행한다.  
- **RunnableSequence**: Runnable들의 시퀀스를 정의한다.  
- **RunnableLambda**: 파이썬 함수를 래핑해 Runnable로 만든다.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')


### RunnableLambda  
일반 Python 함수를 lcel 체인에서 사용할 수 있는 Runnable 형태로 wrapping 처리해주는 클래스

In [ ]:
# 입력을 받아 내장된 함수를 실행하는 Runnable
from langchain_core.runnables import RunnableLambda

runnable = RunnableLambda(lambda x: len(x))
runnable.invoke('안녕 만나서 반갑다~')

11

In [ ]:
# batch() : 여러 건의 입력을 일괄처리해줌
runnable.batch(['안녕 만나서 반갑다~','너도? 나도','?!','🐽🐽🐽🐽🐽🐽'])

[11, 6, 2, 6]

In [5]:
def celsius_to_fahrenheit(celsius):
    return celsius * 9 / 5 + 32

celsius_temps = [0,25,100,-10,37]
runnable = RunnableLambda(celsius_to_fahrenheit)
runnable.batch(celsius_temps)

[32.0, 77.0, 212.0, 14.0, 98.6]

In [ ]:
import time     # 출력 딜레이용

def generator(x):
    for y in x: # 입력을 문자 단위로 순회
        yield y # 한 글자씩 반환

runnable = RunnableLambda(generator)
for chunk in runnable.stream('안녕하세요~😀😀😀😀😀안녕하세요~😀😀😀😀😀안녕하세요~😀😀😀😀😀안녕하세요~😀😀😀😀😀'):
    print(chunk, end='',flush=True) # chunk를 줄바꿈없이 즉시 출력
    time.sleep(0.1)     # 글자 출력마다 딜레이 0.1초

안녕하세요~😀😀😀😀😀안녕하세요~😀😀😀😀😀안녕하세요~😀😀😀😀😀안녕하세요~😀😀😀😀😀

In [7]:
# 사용 예시
def gen(x):
    for y in x:
        yield y

gen10 = gen(range(10))

for n in gen10:
    print(n)

0
1
2
3
4
5
6
7
8
9


In [ ]:
next(gen10) # 제너레이터 다음 값 1개 반환(다 꺼내고 나면 StopIteration 발생)

StopIteration: 

### RunnableSequence
Runnable 객체를 순차연결해주는 Runnable 객체

In [ ]:
from langchain_core.runnables import RunnableSequence   # Runnable들을 순서대로 연결하는 시퀀스

runnable1 = RunnableLambda(lambda x: {'foo':x})
runnable2 = RunnableLambda(lambda x: [x]*3)

chain = RunnableSequence(runnable1,runnable2)
chain.invoke(3)

[{'foo': 3}, {'foo': 3}, {'foo': 3}]

In [10]:
chain = runnable1 | runnable2
chain.invoke(3)

[{'foo': 3}, {'foo': 3}, {'foo': 3}]

### RunnableParallel
여러 Runnable 객체를 인자로 받아, 병렬처리 후 각각의 응답을 하나의 dict로 반환

In [11]:
from langchain_core.runnables import RunnableParallel   # 여러 Runnable들을 같은 입력으로 병렬로 실행

runnable1 = RunnableLambda(lambda x: {'foo':x})
runnable2 = RunnableLambda(lambda x: [x]*3)

chain = RunnableParallel(r1 = runnable1,r2 = runnable2) # r1, r2를 병렬 실행해 dict로 반환
chain.invoke(3)

{'r1': {'foo': 3}, 'r2': [3, 3, 3]}

- 사용자가 준 주제를 이용해 삼행시,농담,시를 각각 생성해서 하나의 응답으로 반환

In [13]:
from langchain_core.prompts import PromptTemplate       # prompt chain 구성
from langchain.chat_models import init_chat_model       # 모델 chain  구성 래퍼
from langchain_core.output_parsers import StrOutputParser   # 답변 문자형 변환
from langchain_core.runnables import RunnableParallel

prompt = PromptTemplate.from_template('{city}의 특산물은 무엇입니까?')
llm = init_chat_model('gpt-5.6-luna')
output_parser = StrOutputParser()

acrostic_poem_prompt = PromptTemplate.from_template(
    '당신은 n행시의 엄청난 고수입니다. 다음 주제로 n행시를 지어주세요. 주제 : {topic}'
)
n_poem_chain = acrostic_poem_prompt | llm | output_parser

joke_prompt = PromptTemplate.from_template(
    '당신은 한국시 농담계의 엄청난 고수입니다. 다음 주제로 배꼽이 빠질만한 농담을 지어주세요. 주제 : {topic}'
)
joke_chain = joke_prompt|llm|output_parser

poem_prompt = PromptTemplate.from_template(
    '당신은 엄청난 현대시의 작가입니다. 다음 주제로 눈물이 나올 정도의 감성적인 시를 지어주세요. 주제 : {topic}'
)
poem_chain = poem_prompt|llm|output_parser

chain = RunnableParallel(
    acrostic_poem = n_poem_chain,
    joke = joke_chain,
    poem = poem_chain
)

def combine_result(input_dict : dict) -> str:
    acrostic_poem = input_dict['acrostic_poem']
    joke = input_dict['joke']
    poem = input_dict['poem']
    return f"""
    n행시:
    {acrostic_poem}

    농담:
    {joke}

    현대시 :
    {poem}
    """

chain = chain | RunnableLambda(combine_result)
print(chain.invoke({'topic' : '아저씨'}))


    n행시:
    **아**무리 마음은 청춘이라 우겨도  
**저**녁 9시면 슬슬 졸음이 오고  
**씨**익 웃으며 말하지, “내일은 꼭 운동한다!”

    농담:
    아저씨가 제일 좋아하는 음료는?

**아이스 아메리카노.**  
왜냐하면 “아, 이스 아메리카노?” 하고 한 번 더 물어보게 하거든. 😄

---

아저씨가 사진관에 가서 말했대.

“사진 좀 잘 나오게 찍어주세요.”

사진사가 “어떤 느낌으로요?” 하니까,

“**월급날 통장 잔액을 본 사람처럼… 현실감 있게요.**” 😭

---

아저씨의 최애 운동은?

**눈치 보기.**  
회사에서는 하루 종일 하고, 집에서는 퇴근 후에도 계속한대.

    현대시 :
    ### 아저씨

아저씨는  
늘 늦은 저녁에만 나타났다.

시장에서 돌아오는 길,  
한 손에는 식은 붕어빵 봉지가 들려 있었고  
다른 손은 아무것도 들지 않은 채  
주머니 속에서 오래 굳어 있었다.

“먹어라.”  
그가 내민 붕어빵은  
언제나 조금 찌그러져 있었지만  
이상하게도 가장 따뜻했다.

나는 자라면서  
그가 왜 웃지 않는지 몰랐다.  
왜 새벽마다 기침을 삼키며  
현관문을 조용히 닫는지도 몰랐다.

다만 비가 오면  
우산을 내 쪽으로 더 기울이던 사람,  
내 신발끈을 묶어주고는  
자기 신발은 젖은 채 걷던 사람,  
내가 넘어질 때마다  
“괜찮다”는 말을 먼저 해주던 사람.

아저씨가 떠난 뒤에야 알았다.

그가 내게 건넨 것은  
붕어빵이 아니라  
자신의 저녁이었고,

내게 씌워준 것은  
우산이 아니라  
평생 젖어도 좋다는 마음이었다.

오늘도 길에서  
비슷한 뒷모습을 보면  
나는 한참을 멈춰 선다.

혹시나 돌아보며  
“많이 컸네.”  
그 한마디 해줄까 봐.

하지만 세상에는  
다시는 돌아보지 않는 사람들이 있고,

남겨진 사람은  
그들의 뒷모습을 닮아가며  
조금씩 어른이 된다.
    


### RunnablePassThrough  
- 사용자의 입력값을 그대로 전달해주는 Runnable

In [14]:
acrostic_poem_prompt = PromptTemplate.from_template(
    '당신은 n행시의 엄청난 고수입니다. 다음 주제로 n행시를 지어주세요. 주제 : {topic}'
)
n_poem_chain = acrostic_poem_prompt | llm | output_parser
print(n_poem_chain.invoke({'topic' : '텀블러'}))


텀: 텀을 두고 천천히 마셔도  
블: 블랙커피의 향기는 오래 남고  
러: 러브까지 담아 다니는 나만의 텀블러


In [16]:
from langchain_core.runnables import RunnablePassthrough
prompt = PromptTemplate.from_template(
    '당신은 n행시의 엄청난 고수입니다. 다음 주제로 n행시를 지어주세요. 주제 : {topic}'
)
chain = {"topic": RunnablePassthrough()}| prompt | llm | output_parser
print(chain.invoke({'topic' : '네트워크'}))

네: 네가 보낸 작은 신호 하나가  
트: 트인 세상 곳곳으로 퍼져 나가  
워: 워낙 멀리 있어도 마음을 이어 주니  
크: 크게 보면 우리는 모두 하나의 네트워크!


In [ ]:
prompt = PromptTemplate.from_template('''
당신은 n행시의 엄청난 고수입니다. 다음 주제로 {n}행시를 지어주세요. \
주제 : {topic}

출력형식
==== <주제> <n행시> ====
<n행시 작성>    
''')
chain = ({"topic": RunnablePassthrough()}
        | RunnablePassthrough.assign(
            n=lambda x: len(x['topic']),
            k = lambda x:100
        )
        | prompt            # 확장된 dict(topic,n,k)를 프롬프트에 주입하여 완성
        | llm 
        | output_parser)


print(chain.invoke('바보'))

==== 바보 2행시 ====
바: 바보인 줄 알았는데,  
보: 보석처럼 빛나는 매력이 있네!
